In [ ]:
# ============================================================
# CROSS-VALIDATION + DATA LEAKAGE - COMPLETE COLAB CODE
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    GridSearchCV,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score


# ============================================================
# 2. LOAD DATASET
# ============================================================

data = load_breast_cancer()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = pd.Series(
    data.target,
    name="target"
)

print("Dataset shape:", X.shape)

print("\nFirst 5 rows:")
print(X.head())


# ============================================================
# 3. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)


# ============================================================
# 4. DATA LEAKAGE DEMO
# ============================================================

# ------------------------------------------------------------
# WRONG METHOD:
# Fit scaler on the complete dataset BEFORE splitting
# ------------------------------------------------------------

scaler_wrong = StandardScaler()

X_scaled_wrong = scaler_wrong.fit_transform(
    X
)

X_train_wrong, X_test_wrong, y_train_wrong, y_test_wrong = train_test_split(
    X_scaled_wrong,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

wrong_model = LogisticRegression(
    max_iter=5000,
    random_state=42
)

wrong_model.fit(
    X_train_wrong,
    y_train_wrong
)

wrong_prediction = wrong_model.predict(
    X_test_wrong
)

wrong_accuracy = accuracy_score(
    y_test_wrong,
    wrong_prediction
)


# ------------------------------------------------------------
# CORRECT METHOD:
# Split first, then fit scaler only on training data
# ------------------------------------------------------------

scaler_correct = StandardScaler()

X_train_scaled = scaler_correct.fit_transform(
    X_train
)

X_test_scaled = scaler_correct.transform(
    X_test
)

correct_model = LogisticRegression(
    max_iter=5000,
    random_state=42
)

correct_model.fit(
    X_train_scaled,
    y_train
)

correct_prediction = correct_model.predict(
    X_test_scaled
)

correct_accuracy = accuracy_score(
    y_test,
    correct_prediction
)


# ------------------------------------------------------------
# Display leakage comparison
# ------------------------------------------------------------

print("\n================================================")
print("DATA LEAKAGE DEMO")
print("================================================")

print(
    "Scaler fitted BEFORE split accuracy:",
    wrong_accuracy
)

print(
    "Scaler fitted AFTER split accuracy:",
    correct_accuracy
)


# ============================================================
# 5. STRATIFIED K-FOLD
# ============================================================

stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=5000,
            random_state=42
        )
    )
])

stratified_scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=stratified_kfold,
    scoring="accuracy"
)

print("\n================================================")
print("STRATIFIED K-FOLD")
print("================================================")

print(
    "Scores:",
    stratified_scores
)

print(
    "Mean:",
    stratified_scores.mean()
)

print(
    "Standard Deviation:",
    stratified_scores.std()
)


# ============================================================
# 6. NORMAL K-FOLD
# ============================================================

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

kfold_scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=kfold,
    scoring="accuracy"
)

print("\n================================================")
print("K-FOLD")
print("================================================")

print(
    "Scores:",
    kfold_scores
)

print(
    "Mean:",
    kfold_scores.mean()
)

print(
    "Standard Deviation:",
    kfold_scores.std()
)


# ============================================================
# 7. STRATIFIED K-FOLD VS K-FOLD COMPARISON
# ============================================================

cv_comparison = pd.DataFrame({
    "Fold": [1, 2, 3, 4, 5],
    "StratifiedKFold": stratified_scores,
    "KFold": kfold_scores
})

print("\n================================================")
print("STRATIFIED K-FOLD VS K-FOLD")
print("================================================")

print(cv_comparison)

print("\nStratifiedKFold Mean:",
      stratified_scores.mean())

print("StratifiedKFold Std:",
      stratified_scores.std())

print("\nKFold Mean:",
      kfold_scores.mean())

print("KFold Std:",
      kfold_scores.std())


# ============================================================
# 8. PARAMETER GRID
# ============================================================

param_grid = {
    "model__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ]
}


# ============================================================
# 9. NON-NESTED CROSS-VALIDATION
# ============================================================

non_nested_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=non_nested_cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(
    X,
    y
)

non_nested_best = grid_search.best_score_

best_params = grid_search.best_params_


print("\n================================================")
print("NON-NESTED CROSS-VALIDATION")
print("================================================")

print(
    "Best CV Score:",
    non_nested_best
)

print(
    "Best Parameters:",
    best_params
)


# ============================================================
# 10. NESTED CROSS-VALIDATION
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

nested_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="accuracy",
    n_jobs=-1
)

nested_scores = cross_val_score(
    nested_grid,
    X,
    y,
    cv=outer_cv,
    scoring="accuracy",
    n_jobs=-1
)

nested_cv_mean = nested_scores.mean()

nested_cv_std = nested_scores.std()


print("\n================================================")
print("NESTED CROSS-VALIDATION")
print("================================================")

print(
    "Nested CV Scores:",
    nested_scores
)

print(
    "Nested CV Mean:",
    nested_cv_mean
)

print(
    "Nested CV Std:",
    nested_cv_std
)


# ============================================================
# 11. NESTED CV OPTIMISM BIAS
# ============================================================

optimism_bias = (
    non_nested_best -
    nested_cv_mean
)

print("\n================================================")
print("NESTED CV OPTIMISM BIAS")
print("================================================")

print(
    "Non-Nested Best Score:",
    non_nested_best
)

print(
    "Nested CV Mean:",
    nested_cv_mean
)

print(
    "Optimism Bias:",
    optimism_bias
)


# ============================================================
# 12. FULL_CV_REPORT() FUNCTION
# ============================================================

def full_cv_report(
    estimator,
    X,
    y,
    param_grid,
    outer_k=5,
    inner_k=3
):

    # --------------------------------------------------------
    # Stratified K-Fold
    # --------------------------------------------------------

    stratified_cv = StratifiedKFold(
        n_splits=outer_k,
        shuffle=True,
        random_state=42
    )

    stratified_scores = cross_val_score(
        estimator,
        X,
        y,
        cv=stratified_cv,
        scoring="accuracy",
        n_jobs=-1
    )

    stratified_kfold_mean = (
        stratified_scores.mean()
    )

    stratified_kfold_std = (
        stratified_scores.std()
    )


    # --------------------------------------------------------
    # Inner CV
    # --------------------------------------------------------

    inner_cv = StratifiedKFold(
        n_splits=inner_k,
        shuffle=True,
        random_state=42
    )


    # --------------------------------------------------------
    # Outer CV
    # --------------------------------------------------------

    outer_cv = StratifiedKFold(
        n_splits=outer_k,
        shuffle=True,
        random_state=42
    )


    # --------------------------------------------------------
    # Nested CV
    # --------------------------------------------------------

    nested_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=inner_cv,
        scoring="accuracy",
        n_jobs=-1
    )

    nested_scores = cross_val_score(
        nested_search,
        X,
        y,
        cv=outer_cv,
        scoring="accuracy",
        n_jobs=-1
    )

    nested_cv_mean = (
        nested_scores.mean()
    )


    # --------------------------------------------------------
    # Non-Nested CV
    # --------------------------------------------------------

    non_nested_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=outer_cv,
        scoring="accuracy",
        n_jobs=-1
    )

    non_nested_search.fit(
        X,
        y
    )

    non_nested_best = (
        non_nested_search.best_score_
    )


    # --------------------------------------------------------
    # Optimism Bias
    # --------------------------------------------------------

    optimism_bias = (
        non_nested_best -
        nested_cv_mean
    )


    # --------------------------------------------------------
    # Return dictionary
    # --------------------------------------------------------

    return {
        "stratified_kfold_mean":
            stratified_kfold_mean,

        "std":
            stratified_kfold_std,

        "nested_cv_mean":
            nested_cv_mean,

        "non_nested_best":
            non_nested_best,

        "optimism_bias":
            optimism_bias
    }


# ============================================================
# 13. RUN FULL_CV_REPORT()
# ============================================================

report = full_cv_report(
    estimator=pipeline,
    X=X,
    y=y,
    param_grid=param_grid,
    outer_k=5,
    inner_k=3
)

print("\n================================================")
print("FULL CV REPORT")
print("================================================")

print(
    "Stratified K-Fold Mean:",
    report["stratified_kfold_mean"]
)

print(
    "Standard Deviation:",
    report["std"]
)

print(
    "Nested CV Mean:",
    report["nested_cv_mean"]
)

print(
    "Non-Nested Best:",
    report["non_nested_best"]
)

print(
    "Optimism Bias:",
    report["optimism_bias"]
)


# ============================================================
# 14. DISPLAY REPORT AS DATAFRAME
# ============================================================

report_df = pd.DataFrame({
    "Metric": [
        "Stratified K-Fold Mean",
        "Standard Deviation",
        "Nested CV Mean",
        "Non-Nested Best",
        "Optimism Bias"
    ],

    "Value": [
        report["stratified_kfold_mean"],
        report["std"],
        report["nested_cv_mean"],
        report["non_nested_best"],
        report["optimism_bias"]
    ]
})

print("\nFinal CV Report:")
print(report_df)


# ============================================================
# END OF CODE
# ============================================================

Dataset shape: (569, 30)

First 5 rows:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst radius  worst 